# BDA
## ORM con SQLAlchemy: la BD Docencia como clases Python

En `sql_sqlalchemy.ipynb` las consultas se escriben en SQL dentro de `text()`. El **ORM** (*Object-Relational Mapping*) de SQLAlchemy da un paso más: cada tabla se representa con una **clase Python**, cada fila con un **objeto** y cada clave ajena con un **atributo** que lleva de un objeto a otro (`profesor.departamento`). SQLAlchemy escribe el SQL.

En este notebook:

1. Las cuatro tablas de Docencia como clases: columnas, claves y restricciones.
2. Leer objetos y navegar por las relaciones.
3. Insertar, modificar y borrar con una `Session`, y qué pasa al incumplir una restricción.
4. Crear las tablas a partir de las clases.

Todo lo que se inserta en el apartado 3 se borra al final del apartado: la BD queda como estaba.

---

## Instalación de dependencias

In [ ]:
# Instalar dependencias necesarias
#!pip install sqlalchemy mysql-connector-python psycopg2-binary

## Importar librerías

In [ ]:
from contextlib import contextmanager
from typing import Optional

import sqlalchemy
from sqlalchemy import (CHAR, CheckConstraint, Float, ForeignKey, Integer, create_engine,
                        event, exc, func, inspect, select, text)
from sqlalchemy.engine import URL
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column, relationship
from sqlalchemy.types import TypeDecorator

print(f"SQLAlchemy {sqlalchemy.__version__}")

## Configuración de Conexión

In [ ]:
# ===========================================================================
# CONFIGURACIÓN - Edita estos valores
# ===========================================================================

# Selecciona el tipo de base de datos: 'sqlite', 'mariadb' o 'postgres'
TIPO_BD = 'sqlite'

# Configuración para SQLite
SQLITE_FILE = '../dbs/docencia.db'

# Configuración para MariaDB/MySQL
MARIADB_HOST = 'localhost'
MARIADB_PORT = 3306
MARIADB_DATABASE = 'docencia'
MARIADB_USER = 'masteruser'
MARIADB_PASSWORD = 'masterpassword'

# Configuración para PostgreSQL
POSTGRES_HOST = 'localhost'
POSTGRES_PORT = 5432
POSTGRES_DATABASE = 'master'
POSTGRES_USER = 'masteruser'
POSTGRES_PASSWORD = 'masterpassword'
POSTGRES_SCHEMA = 'docencia'

print(f"Configuración lista para: {TIPO_BD.upper()}")

## Crear el *engine*

Igual que en `sql_sqlalchemy.ipynb`. Además, `ver_sql()` permite ver el SQL que SQLAlchemy envía al SGBD: todo lo que se ejecute dentro de un bloque `with ver_sql():` se muestra con el prefijo `SQL>`.

In [ ]:
if TIPO_BD == 'sqlite':
    engine = create_engine(f"sqlite:///{SQLITE_FILE}")

    # SQLite no comprueba las claves ajenas si no se activan en cada conexión
    @event.listens_for(engine, "connect")
    def activar_claves_ajenas(dbapi_conn, connection_record):
        dbapi_conn.execute("PRAGMA foreign_keys = ON")

elif TIPO_BD == 'mariadb':
    url = URL.create("mysql+mysqlconnector",
                     username=MARIADB_USER, password=MARIADB_PASSWORD,
                     host=MARIADB_HOST, port=MARIADB_PORT, database=MARIADB_DATABASE)
    engine = create_engine(url)

elif TIPO_BD == 'postgres':
    url = URL.create("postgresql+psycopg2",
                     username=POSTGRES_USER, password=POSTGRES_PASSWORD,
                     host=POSTGRES_HOST, port=POSTGRES_PORT, database=POSTGRES_DATABASE)
    # search_path: las tablas se buscan en el esquema docencia sin escribir el prefijo
    engine = create_engine(url, connect_args={"options": f"-c search_path={POSTGRES_SCHEMA}"})

else:
    raise ValueError(f"Tipo de base de datos '{TIPO_BD}' no válido. Usa: 'sqlite', 'mariadb' o 'postgres'")

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))

print(f"Conectado a {engine.dialect.name} ({engine.url.render_as_string(hide_password=True)})")

# Muestra cada sentencia que se envía al SGBD mientras VER_SQL es True
VER_SQL = True

@event.listens_for(engine, "before_cursor_execute")
def mostrar_sql(conn, cursor, statement, parameters, context, executemany):
    if VER_SQL:
        print(f"SQL> {' '.join(statement.split())}   {parameters}")

@contextmanager
def ver_sql():
    global VER_SQL
    VER_SQL = True
    try:
        yield
    finally:
        VER_SQL = False

---

## 1. Las tablas como clases

Cada clase hereda de una clase base común (`Base`) y declara:

| En SQL | En la clase |
|---|---|
| nombre de la tabla | `__tablename__` |
| columna | atributo con `mapped_column(tipo)` |
| `NOT NULL` (VNN) | `Mapped[str]`; si admite nulos, `Mapped[Optional[str]]` |
| clave primaria (CP) | `primary_key=True` |
| `UNIQUE` (Uni) | `unique=True` |
| clave ajena (CAj) | `ForeignKey("Tabla.columna")` |
| `CHECK` (RI) | `CheckConstraint(...)` en `__table_args__` |

Hay dos detalles para que las mismas clases funcionen en los tres SGBD:

- **Nombres de tabla**: MariaDB distingue mayúsculas (la tabla es `Departamento`) y PostgreSQL las tiene en minúsculas (`departamento`). SQLAlchemy escribe entre comillas los nombres con mayúsculas, y entonces PostgreSQL buscaría `"Departamento"` exactamente. Con `"quote": False` el nombre va sin comillas y funciona en los tres.
- **`CHAR(n)`**: PostgreSQL devuelve el valor relleno con espacios hasta `n` caracteres (`'01'` → `'01       '`). `Texto` es un tipo propio: en la BD es `CHAR(n)`, pero al leer quita los espacios del final.

In [ ]:
class Texto(TypeDecorator):
    # CHAR(n) en la BD; al leer, sin los espacios de relleno
    impl = CHAR
    cache_ok = True

    def process_result_value(self, value, dialect):
        return value.rstrip() if value is not None else None


class Base(DeclarativeBase):
    pass

### Departamento

`Departamento` y `Profesor` están unidas por **dos** claves ajenas: cada profesor pertenece a un departamento (`Profesor.cod_dep`) y cada departamento tiene un director (`Departamento.director`). Como hay dos caminos entre las mismas tablas, cada relación indica con `foreign_keys` cuál usa.

- `profesores` y `asignaturas` son listas: los objetos del otro lado de una relación 1:N.
- La columna `director` guarda el DNI; en la clase se llama `dni_director`, y el nombre `director` se reserva para la relación, que devuelve el objeto `Profesor`.
- `post_update=True`: un departamento y su director se necesitan mutuamente (cada uno tiene una clave ajena hacia el otro). Al guardar los dos a la vez, SQLAlchemy inserta primero ambos y después pone el director con un `UPDATE`.

In [ ]:
class Departamento(Base):
    __tablename__ = "Departamento"
    __table_args__ = {"quote": False}

    cod_dep: Mapped[str] = mapped_column(Texto(4), primary_key=True)                 # CP
    nombre: Mapped[str] = mapped_column(Texto(50))                                   # VNN
    telefono: Mapped[Optional[str]] = mapped_column(Texto(8))
    dni_director: Mapped[Optional[str]] = mapped_column(
        "director", Texto(9),
        ForeignKey("Profesor.dni", ondelete="SET NULL", use_alter=True, name="fk_dep_director"))  # CAj

    profesores: Mapped[list["Profesor"]] = relationship(
        back_populates="departamento", foreign_keys="Profesor.cod_dep")
    director: Mapped[Optional["Profesor"]] = relationship(foreign_keys=[dni_director], post_update=True)
    asignaturas: Mapped[list["Asignatura"]] = relationship(back_populates="departamento")

    def __repr__(self):
        return f"Departamento({self.cod_dep!r}, {self.nombre!r})"

### Profesor

`back_populates` une las dos puntas de una relación: `profesor.departamento` y `departamento.profesores` se mantienen sincronizadas.

En la BD, al borrar un profesor se borra su docencia (`ON DELETE CASCADE`). La relación `docencias` hace lo mismo en Python: `cascade="all, delete-orphan"` borra la docencia junto con el profesor, y `passive_deletes=True` deja que, si no está cargada, la borre el propio SGBD.

In [ ]:
class Profesor(Base):
    __tablename__ = "Profesor"
    __table_args__ = {"quote": False}

    dni: Mapped[str] = mapped_column(Texto(9), primary_key=True)                     # CP
    nombre: Mapped[str] = mapped_column(Texto(80))                                   # VNN
    telefono: Mapped[Optional[str]] = mapped_column(Texto(8))
    cod_dep: Mapped[str] = mapped_column(Texto(4), ForeignKey("Departamento.cod_dep"))  # VNN y CAj
    provincia: Mapped[Optional[str]] = mapped_column(Texto(25))
    edad: Mapped[Optional[int]] = mapped_column(Integer)

    departamento: Mapped["Departamento"] = relationship(back_populates="profesores", foreign_keys=[cod_dep])
    docencias: Mapped[list["Docencia"]] = relationship(
        back_populates="profesor", cascade="all, delete-orphan", passive_deletes=True)

    def __repr__(self):
        return f"Profesor({self.dni!r}, {self.nombre!r})"

### Asignatura

Las restricciones de integridad RI1 y RI2 son `CHECK` de la tabla: van en `__table_args__`, con el mismo nombre que en el script SQL.

En la BD, una asignatura con docencia no se puede borrar (`ON DELETE RESTRICT`). `passive_deletes="all"` le dice a SQLAlchemy que no toque la docencia al borrar la asignatura y deje decidir al SGBD.

In [ ]:
class Asignatura(Base):
    __tablename__ = "Asignatura"
    __table_args__ = (
        CheckConstraint("teoria >= practicas", name="ri1_teoria_practicas"),                          # RI1
        CheckConstraint("semestre IN ('1A','1B','2A','2B','3A','3B','4A','4B')", name="ri2_semestre"),  # RI2
        {"quote": False},
    )

    cod_asg: Mapped[str] = mapped_column(Texto(5), primary_key=True)                 # CP
    nombre: Mapped[str] = mapped_column(Texto(50), unique=True)                      # VNN y Uni
    semestre: Mapped[str] = mapped_column(Texto(2))                                  # VNN
    cod_dep: Mapped[str] = mapped_column(Texto(4), ForeignKey("Departamento.cod_dep"))  # VNN y CAj
    teoria: Mapped[float] = mapped_column(Float)                                     # VNN
    practicas: Mapped[float] = mapped_column(Float)                                  # VNN

    departamento: Mapped["Departamento"] = relationship(back_populates="asignaturas")
    docencias: Mapped[list["Docencia"]] = relationship(back_populates="asignatura", passive_deletes="all")

    def __repr__(self):
        return f"Asignatura({self.cod_asg!r}, {self.nombre!r})"

### Docencia

`Docencia` relaciona profesores y asignaturas (N:M) y además guarda datos propios (`gteo`, `gpra`). Por eso es una clase más, con una clave primaria de dos columnas, y no una simple tabla intermedia.

In [ ]:
class Docencia(Base):
    __tablename__ = "Docencia"
    __table_args__ = {"quote": False}

    dni: Mapped[str] = mapped_column(
        Texto(9), ForeignKey("Profesor.dni", ondelete="CASCADE"), primary_key=True)         # CP y CAj
    cod_asg: Mapped[str] = mapped_column(
        Texto(5), ForeignKey("Asignatura.cod_asg", ondelete="RESTRICT"), primary_key=True)  # CP y CAj
    gteo: Mapped[int] = mapped_column(Integer)                                       # VNN
    gpra: Mapped[int] = mapped_column(Integer)                                       # VNN

    profesor: Mapped["Profesor"] = relationship(back_populates="docencias")
    asignatura: Mapped["Asignatura"] = relationship(back_populates="docencias")

    def __repr__(self):
        return f"Docencia({self.dni!r}, {self.cod_asg!r}, gteo={self.gteo}, gpra={self.gpra})"

---

## 2. Leer objetos y navegar por las relaciones

Una `Session` es el espacio de trabajo con la BD: carga objetos, recuerda sus cambios y los guarda al hacer `commit()`.

`session.get(Clase, clave)` busca una fila por su clave primaria y la devuelve como objeto.

In [ ]:
session = Session(engine)

informatica = session.get(Departamento, "DINF")
print(informatica)
print("Teléfono:", informatica.telefono)
print("Director:", informatica.director)
print("Profesores:")
for p in informatica.profesores:
    print(f"  {p.nombre} ({p.edad} años)")
print("Asignaturas:", [a.nombre for a in informatica.asignaturas])

### Carga perezosa

Los objetos relacionados no se leen hasta que se usan: cada relación que se recorre por primera vez lanza un `SELECT`. Además, la sesión recuerda los objetos que ya ha cargado (*identity map*): pedir otra vez un objeto conocido no consulta la BD.

En la celda, Luis Garcia ya se cargó al recorrer `informatica.profesores`, así que `session.get` no lanza ningún SQL. Sus docencias sí se leen de la BD; sus asignaturas, no, porque ya se cargaron con `informatica.asignaturas`.

In [ ]:
with ver_sql():
    luis = session.get(Profesor, "02")
    print(luis, "->", luis.departamento.nombre)
    for d in luis.docencias:
        print(f"  {d.asignatura.nombre}: {d.gteo} grupos de teoría y {d.gpra} de prácticas")

### Consultas con `select()`

También se pueden buscar objetos con una condición. `select(Profesor)` devuelve objetos `Profesor` completos, y las condiciones se escriben con los atributos de la clase. El notebook `sql_orm_imdb.ipynb` trata las consultas a fondo.

In [ ]:
consulta = select(Profesor).where(Profesor.edad > 45).order_by(Profesor.edad.desc())
print(consulta)
print()

for p in session.scalars(consulta):
    print(f"{p.nombre} ({p.edad} años), {p.departamento.nombre}")

---

## 3. Insertar, modificar y borrar

Los cambios se hacen sobre los objetos y la sesión los traduce a `INSERT`, `UPDATE` y `DELETE` al hacer `commit()`. Si algo falla, `rollback()` deshace todo lo pendiente.

Antes de empezar se cuentan las filas de cada tabla, para comprobar al final que la BD queda igual.

In [ ]:
def contar_filas():
    return {clase.__name__: session.scalar(select(func.count()).select_from(clase))
            for clase in (Departamento, Profesor, Asignatura, Docencia)}

filas_antes = contar_filas()
print(filas_antes)

### Insertar

Se crea el objeto como cualquier objeto Python, se añade a la sesión con `add()` y se guarda con `commit()`. Antes del `commit()`, el objeto está en `session.new`: pendiente de insertar.

In [ ]:
prueba = Profesor(dni="99", nombre="Profesor de Prueba", cod_dep="DINF", edad=30)
session.add(prueba)
print("Pendientes de insertar:", session.new)

with ver_sql():
    session.commit()

Los objetos relacionados se pueden crear a la vez. Al añadir una `Docencia` a `prueba.docencias`, la sesión guarda también la asignatura nueva, sin `add()`, y las inserta en el orden que exigen las claves ajenas: primero la asignatura y después la docencia.

In [ ]:
asig_prueba = Asignatura(cod_asg="X01", nombre="Asignatura de prueba", semestre="1A",
                         cod_dep="DINF", teoria=3, practicas=1)
prueba.docencias.append(Docencia(asignatura=asig_prueba, gteo=1, gpra=2))

with ver_sql():
    session.commit()

print(prueba.docencias)
print("Profesores de Informática:", [p.nombre for p in informatica.profesores])

### Modificar

Basta con cambiar los atributos. La sesión detecta qué objetos han cambiado (`session.dirty`) y solo actualiza las columnas modificadas.

In [ ]:
prueba.edad = 31
prueba.provincia = "València"
print("Modificados:", session.dirty)

with ver_sql():
    session.commit()

### Incumplir una restricción

SQLAlchemy no comprueba las restricciones: las comprueba el SGBD al hacer `commit()`. Si falla, la sesión queda a la espera de un `rollback()`, que descarta los cambios pendientes.

Esta asignatura incumple RI1 (`teoria >= practicas`):

In [ ]:
session.add(Asignatura(cod_asg="X02", nombre="Asignatura mala", semestre="1A",
                       cod_dep="DINF", teoria=1, practicas=3))
try:
    session.commit()
except exc.DBAPIError as e:
    session.rollback()
    print(f"{type(e).__name__}: {e.orig}")

Y este profesor tiene un departamento que no existe (clave ajena):

In [ ]:
session.add(Profesor(dni="98", nombre="Profesor sin departamento", cod_dep="XXXX"))
try:
    session.commit()
except exc.DBAPIError as e:
    session.rollback()
    print(f"{type(e).__name__}: {e.orig}")

### Borrar

`session.delete(objeto)` borra la fila al hacer `commit()`, y las relaciones deciden qué pasa con los objetos que dependen de ella.

La asignatura de prueba tiene docencia, y la BD no permite borrarla (`ON DELETE RESTRICT`):

In [ ]:
session.delete(asig_prueba)
try:
    session.commit()
except exc.DBAPIError as e:
    session.rollback()
    print(f"{type(e).__name__}: {e.orig}")

Al borrar el profesor, en cambio, su docencia se borra con él. Como no está cargada en la sesión, SQLAlchemy no la lee: solo borra el profesor y es el SGBD quien borra su docencia (`ON DELETE CASCADE`), gracias a `passive_deletes=True`. Si estuviera cargada, la borraría el propio SQLAlchemy (`cascade="all, delete-orphan"`). Después ya se puede borrar la asignatura.

In [ ]:
with ver_sql():
    session.delete(prueba)
    session.commit()

session.delete(asig_prueba)
session.commit()

print("Antes:  ", filas_antes)
print("Después:", contar_filas())
print("La BD queda como estaba" if contar_filas() == filas_antes else "¡La BD ha cambiado!")

In [ ]:
session.close()

---

## 4. Crear las tablas desde las clases

Las clases contienen todo lo necesario para crear las tablas: `Base.metadata.create_all(engine)` lanza un `CREATE TABLE` por cada clase que no exista todavía en la BD. Así el esquema sale del código en lugar de un script SQL.

`create_all` solo crea las tablas que faltan; no modifica las que ya existen. Para cambiar un esquema que ya tiene datos se usan herramientas de migraciones como Alembic.

Primero, el `CREATE TABLE` que se generaría para `Asignatura` en MariaDB y en PostgreSQL, sin conectarse a ninguno. Sale igual en los dos porque todos los tipos de la clase son estándar; con tipos propios de un SGBD, cada dialecto escribiría el suyo.

In [ ]:
from sqlalchemy.dialects import mysql, postgresql
from sqlalchemy.schema import CreateTable

print("MariaDB:", CreateTable(Asignatura.__table__).compile(dialect=mysql.dialect()))
print("PostgreSQL:", CreateTable(Asignatura.__table__).compile(dialect=postgresql.dialect()))

Ahora se crean las cuatro tablas en una BD SQLite **en memoria** (`sqlite://`): existe solo mientras el programa está abierto, así que no deja ningún fichero.

La clave ajena del director (`use_alter=True`) forma un ciclo con la de `Profesor.cod_dep`: cada tabla necesita que exista la otra. En MariaDB y PostgreSQL se añade al final con un `ALTER TABLE`; SQLite no lo necesita.

In [ ]:
memoria = create_engine("sqlite://")

@event.listens_for(memoria, "connect")
def activar_claves_ajenas_memoria(dbapi_conn, connection_record):
    dbapi_conn.execute("PRAGMA foreign_keys = ON")

Base.metadata.create_all(memoria)
print("Tablas creadas:", inspect(memoria).get_table_names())

Se llenan con objetos. El departamento y su directora se necesitan mutuamente, y gracias a `post_update` se guardan los dos en el mismo `commit()`:

In [ ]:
with Session(memoria) as s:
    dep = Departamento(cod_dep="DPRU", nombre="Pruebas")
    ana = Profesor(dni="1", nombre="Ana", edad=40, departamento=dep)
    pau = Profesor(dni="2", nombre="Pau", edad=35, departamento=dep)
    dep.director = ana
    bd = Asignatura(cod_asg="BD", nombre="Bases de datos", semestre="2A",
                    teoria=3, practicas=3, departamento=dep)
    ana.docencias.append(Docencia(asignatura=bd, gteo=2, gpra=1))
    pau.docencias.append(Docencia(asignatura=bd, gteo=0, gpra=3))
    s.add(dep)
    s.commit()

    for d in s.scalars(select(Docencia).order_by(Docencia.dni)):
        print(f"{d.profesor.nombre} imparte {d.asignatura.nombre} "
              f"(director/a de su departamento: {d.profesor.departamento.director.nombre})")

## Cerrar Conexión

In [ ]:
memoria.dispose()
engine.dispose()
print(f"Conexiones a {TIPO_BD.upper()} cerradas correctamente")